# Cálculo de Sondas (Regla de los Doceavos)
Interpolar alturas de marea entre la pleamar y la bajamar usando armónicos simples.

Este simulador está diseñado para fines educativos. **No lo utilices para la navegación real.**

<a href="https://colab.research.google.com/github/jorgejuan007/Nautica/blob/main/simulaciones/63_simulador_marea_regla_doceavos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import numpy as np

def regla_doceavos(hp_str, ap, hb_str, ab, hora_actual_str):
    try:
        # Calcular si sube o baja (Flujo / Reflujo)
        fmt = "%H:%M"
        t1 = datetime.strptime(hp_str, fmt)
        t2 = datetime.strptime(hb_str, fmt)
        t_req = datetime.strptime(hora_actual_str, fmt)
        
        # Ajustar por paso de medianoche si es necesario
        if t2 < t1:
            if t_req < t1:
                t1 -= timedelta(days=1)
            else:
                t2 += timedelta(days=1)
        
        if t_req < t1 or t_req > t2:
            print("Error: La hora solicitada debe estar entre las dos mareas introducidas.")
            return

        duracion_total = (t2 - t1).total_seconds() / 3600
        horas_pasadas = (t_req - t1).total_seconds() / 3600
        
        amplitud = abs(ap - ab)
        
        # Doceavos por cada "hora" de marea (una "hora de marea" es duracion/6)
        hora_marea = duracion_total / 6
        un_doceavo = amplitud / 12
        
        # Curva senoidal de marea real para comparar
        horas_sim = np.linspace(0, duracion_total, 100)
        
        if ap > ab:
            # Vaciante (Reflujo)
            fase = 1
            niveles = ab + (amplitud / 2) * (1 + np.cos(np.pi * (horas_sim / duracion_total)))
            altura_actual_teorica = ab + (amplitud / 2) * (1 + np.cos(np.pi * (horas_pasadas / duracion_total)))
        else:
            # Creciente (Flujo)
            fase = -1
            niveles = ap + (amplitud / 2) * (1 + np.cos(np.pi * (1 - (horas_sim / duracion_total))))
            altura_actual_teorica = ap + (amplitud / 2) * (1 + np.cos(np.pi * (1 - (horas_pasadas / duracion_total))))

        print(f"Duración de la marea: {duracion_total:.2f} horas (Una 'hora de marea' = {hora_marea:.2f} hrs)")
        print(f"Amplitud de la marea: {amplitud:.2f} m (1/12 = {un_doceavo:.2f} m)")
        print(f"Han pasado {horas_pasadas:.2f} horas reales desde la marea inicial.")
        print(f"\nAltura de marea estimada a las {hora_actual_str}: {altura_actual_teorica:.2f} metros")
        
        plt.figure(figsize=(8,4))
        plt.plot(horas_sim, niveles, 'b-', label='Curva Senoidal de Marea')
        plt.plot(horas_pasadas, altura_actual_teorica, 'ro', label=f'Altura a las {hora_actual_str}')
        plt.title('Simulación de Curva de Marea (Armónico simple)')
        plt.xlabel(f'Horas transcurridas desde las {hp_str}')
        plt.ylabel('Sonda (metros)')
        plt.legend()
        plt.grid(True)
        plt.show()

    except Exception as e:
        print("Error en el formato de hora. Usa HH:MM (ej. 14:30)")

# Interfaces
hp = widgets.Text(value="08:00", description='Hora Inicial:')
ap = widgets.FloatSlider(value=3.5, min=0, max=12, step=0.1, description='Alt. Inicial:')
hb = widgets.Text(value="14:15", description='Hora Final:')
ab = widgets.FloatSlider(value=0.5, min=0, max=12, step=0.1, description='Alt. Final:')
h_req = widgets.Text(value="10:30", description='Hora a Sabet:')

out = widgets.interactive_output(regla_doceavos, {'hp_str': hp, 'ap': ap, 'hb_str': hb, 'ab': ab, 'hora_actual_str': h_req})
display(widgets.VBox([hp, ap, hb, ab, h_req, out]))
